# Email assistant - end-to-end walkthrough

This notebook drives the same `AssistantService` used by the API, CLI and eval runner:

1. process a 36-thread inbox (guard -> triage -> rank -> per-thread LangGraph)
2. inspect a meeting thread: summary, scheduling decision, tentative holds, grounded draft
3. human review: approve, **edit** (learned into memory) and an edit that **leaks** data (blocked)
4. prompt-injection quarantine
5. telemetry and the audit trail

`MODE = "replay"` uses recorded Gemini responses (no API key needed);
`"live"` calls Gemini; `"offline"` uses only the rule-based fallbacks.

In [1]:
import pandas as pd
from email_assistant.config import Settings
from email_assistant.service import AssistantService
from email_assistant.schemas.review import HumanDecision, ReviewResponse
from evals.run import settings_for

MODE = "replay"
svc = AssistantService.create(settings_for(MODE), ephemeral=True)
orc = svc.orchestrator
print(type(svc.deps.llm).__name__, svc.deps.memory.embedder.name, svc.deps.clock.now())

LLMClient gemini-embedding-001-768 2026-11-09 09:00:00+01:00


## 1. Process the inbox (ranked)

In [2]:
items = orc.process_inbox()
pd.DataFrame([{"rank": i.rank, "status": i.status, "category": i.triage.category.value,
               "urgency": i.triage.urgency.value, "by": i.triage.source,
               "from": i.sender[:30], "subject": i.subject[:50]} for i in items])

,rank,status,category,urgency,by,from,subject
0,1,pending_review,action_request,critical,llm,Sarah Lind <sarah.lind@northwi,URGENT: confirm Q4 GPU budget lines before 16:...
1,2,pending_review,customer,critical,llm,Mikael Ström <mikael.strom@roy,"CRITICAL: deposit webhooks failing, players ca..."
2,3,pending_review,action_request,high,llm,Marcus Vale <marcus.vale@gpucl,GPUCloud reserved capacity contract
3,4,pending_review,customer,high,llm,Karin Sjöberg <karin.sjoberg@p,Mina uttag är försenade
4,5,quarantined,security_threat,high,guard,Dana Ruiz <dana.ruiz@syncwave.,Re: Integration timeline
5,6,quarantined,security_threat,high,guard,Dana Ruiz <dana.ruiz@syncwave.,Partnership opportunity with SyncStream
6,7,quarantined,security_threat,high,guard,Insight Analytics Billing <acc,Invoice 7731 - Insight Analytics
7,8,quarantined,security_threat,high,guard,Ops Notes <ops-notes@cfg-share,Fwd: configuration notes
8,9,quarantined,security_threat,high,guard,System Notice <notice@admin-co,Account notice
9,10,quarantined,security_threat,high,guard,Compliance Bot <policy@northwi,Policy update


## 2. A meeting request that conflicts with the calendar

In [3]:
tid = next(t for t in orc.items if t.startswith("t03"))
state = orc.state(tid)
dec = state["scheduling"]
print("decision:", dec.outcome, "| source:", dec.source, "| warnings:", dec.warnings)
for s in dec.alternatives:
    print("  alternative:", s.label("Europe/Stockholm"), "/", s.label("America/New_York"))
for a in state["actions"].values():
    print(f"{a.type.value:18} {a.status.value:17} {a.rationale}")

decision: propose_alternatives | source: llm | warnings: []
  alternative: Tue 10 Nov 16:00-16:30 (CET) / Tue 10 Nov 10:00-10:30 (EST)
  alternative: Tue 10 Nov 16:30-17:00 (CET) / Tue 10 Nov 10:30-11:00 (EST)
  alternative: Wed 11 Nov 15:00-15:30 (CET) / Wed 11 Nov 09:00-09:30 (EST)
apply_labels       executed          Triage labels
calendar_hold      executed          Tentative hold on a proposed alternative
calendar_hold      executed          Tentative hold on a proposed alternative
calendar_hold      executed          Tentative hold on a proposed alternative
send_email         pending_approval  Reply to meeting (medium)


In [4]:
draft = next(a for a in state["actions"].values() if a.type.value == "send_email")
print(draft.payload.body)
print("\nungrounded claims:", draft.payload.ungrounded_claims, "| source:", draft.payload.draft_source)

Hi Daniel,

thanks for reaching out. I cannot make that time work, but I would be happy to discuss the Q1 roadmap and shared SDK at one of these times: Tue 10 Nov 16:00-16:30 CET (10:00-10:30 EST), Tue 10 Nov 16:30-17:00 CET (10:30-11:00 EST), or Wed 11 Nov 15:00-15:30 CET (09:00-09:30 EST).

Let me know what works for you.

Best,
Alex

ungrounded claims: [] | source: llm


## 3. Human review
Approve the meeting reply; then edit a customer reply (the edit is learned).

In [5]:
req = orc.items[tid].review
orc.submit_review(tid, ReviewResponse(decisions=[
    HumanDecision(action_id=a.id, decision="approve") for a in req.actions]))
print(orc.items[tid].status, "| sent:", [m["to"] for m in svc.deps.mail.sent_messages()])
print("awaiting-reply reminders:",
      [r.title for r in svc.deps.reminders.list(thread_id=tid)])

done | sent: [['daniel.reed@acmetech.example']]
awaiting-reply reminders: []


In [6]:
t08 = next(t for t in orc.items if t.startswith("t08"))
email = orc.items[t08].review.actions[0]
print(email.payload.body)
leaky = email.payload.body + "\nPS: our Q4 GPU budget is EUR 350,000."
item = orc.submit_review(t08, ReviewResponse(decisions=[
    HumanDecision(action_id=email.id, decision="edit", edits={"body": leaky})]))
print("\nafter leaky edit ->", item.status, [f.code for f in item.review.actions[0].findings])

Hi Helen,

It is great to hear from you. Regarding your questions, I cannot share details on our internal budget or market rumours.

We remain fully committed to our product roadmap and supporting your team.

Best,
Alex

after leaky edit -> pending_review ['confidential_leak']


In [7]:
better = email.payload.body.replace("Best,\nAlex", "Cheers,\nAlex")
item = orc.submit_review(t08, ReviewResponse(decisions=[
    HumanDecision(action_id=email.id, decision="edit", edits={"body": better},
                  comment="warmer sign-off for customers")]))
print(item.status)
pd.DataFrame([{"kind": r.kind.value, "key": r.key, "content": r.content[:80], "source": r.source.value}
              for r in svc.deps.memory.list() if r.source.value == "hitl_feedback"])

{"ts": "2026-11-08T16:14:52.642+00:00", "level": "ERROR", "logger": "email_assistant", "msg": "node.failed", "run_id": "cebf68ba", "thread_id": "t08-goldcrest-budget-acquisition-rumours", "node": "learn", "event": "node.failed", "error": "CacheMiss('1 embedding(s) not in replay cache (gemini-embedding-001-768)')"}


done


,kind,key,content,source
0,episodic,t03-acmetech-meeting-request-tuesday,"Hi Daniel,\n\nthanks for reaching out. I canno...",hitl_feedback
1,episodic,t08-goldcrest-budget-acquisition-rumours,"Hi Helen,\n\nIt is great to hear from you. Reg...",hitl_feedback


## 4. Prompt injection: quarantined, nothing drafted

In [8]:
t27 = next(t for t in orc.items if t.startswith("t27"))
i = orc.items[t27]
print(i.status, i.guard.risk_score, i.guard.signals)
print([a.type.value for a in i.actions])

quarantined 0.952 ['exfiltration', 'hitl_bypass', 'override_instructions', 'system_override']
['apply_labels', 'quarantine', 'apply_labels']


## 5. Telemetry and audit trail

In [9]:
import json
print(json.dumps(svc.deps.telemetry.summary(), indent=1)[:2500])

{
 "node_latency_ms": {
  "archiver": {
   "count": 6,
   "p50": 0.0,
   "p95": 0.0
  },
  "drafter": {
   "count": 15,
   "p50": 2.0,
   "p95": 2.2
  },
  "execute": {
   "count": 23,
   "p50": 0.0,
   "p95": 0.0
  },
  "execute_auto": {
   "count": 36,
   "p50": 0.0,
   "p95": 0.1
  },
  "followup": {
   "count": 20,
   "p50": 0.0,
   "p95": 0.3
  },
  "guard": {
   "count": 36,
   "p50": 0.0,
   "p95": 0.0
  },
  "labeler": {
   "count": 36,
   "p50": 0.0,
   "p95": 0.1
  },
  "learn": {
   "count": 23,
   "p50": 0.0,
   "p95": 0.1
  },
  "policy": {
   "count": 36,
   "p50": 0.0,
   "p95": 0.5
  },
  "quarantine": {
   "count": 10,
   "p50": 0.0,
   "p95": 0.1
  },
  "revalidate": {
   "count": 3,
   "p50": 0.2,
   "p95": 0.2
  },
  "review": {
   "count": 19,
   "p50": 0.0,
   "p95": 0.1
  },
  "scheduler": {
   "count": 6,
   "p50": 5.8,
   "p95": 9.8
  },
  "summarizer": {
   "count": 3,
   "p50": 0.4,
   "p95": 0.4
  },
  "supervisor": {
   "count": 132,
   "p50": 0.0,
   "p95"

In [10]:
pd.DataFrame(orc.audit(t08))

,event,plan,category,urgency,source,action,status,findings,ref,decision,comment,override,justification,reviewer
0,plan,"[labeler, followup, drafter]",customer,medium,llm,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,policy,NaN,NaN,NaN,NaN,apply_labels:d4ade6bc69,approved,[],NaN,NaN,NaN,NaN,NaN,NaN
2,policy,NaN,NaN,NaN,NaN,send_email:ca8e5f5503,pending_approval,[],NaN,NaN,NaN,NaN,NaN,NaN
3,executed,NaN,NaN,NaN,NaN,apply_labels:d4ade6bc69,NaN,NaN,labels,NaN,NaN,NaN,NaN,NaN
4,human_decision,NaN,NaN,NaN,NaN,send_email:ca8e5f5503,NaN,NaN,NaN,edit,NaN,False,NaN,owner
5,revalidate,NaN,NaN,NaN,NaN,send_email:ca8e5f5503,pending_approval,[confidential_leak],NaN,NaN,NaN,NaN,NaN,NaN
6,human_decision,NaN,NaN,NaN,NaN,send_email:ca8e5f5503,NaN,NaN,NaN,edit,warmer sign-off for customers,False,NaN,owner
7,executed,NaN,NaN,NaN,NaN,send_email:ca8e5f5503,NaN,NaN,t08-goldcrest-budget-acquisition-rumours-sent-2,NaN,NaN,NaN,NaN,NaN
